In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix
from matplotlib import pyplot as plt
import seaborn as sns
import joblib

In [5]:

# 1. Load your dataset (Make sure it has the raw 'url' string column AND your extracted features!)
data = pd.read_csv('../data/datasets/processed_dataset_v2.csv') 
# Note: change to _v3 or _v4 if that's where your entropy/digits features are saved!

raw_data = pd.read_csv('../data/datasets/raw_dataset.csv')

# 2. Copy the 'url' column over to your current working data
data['url'] = raw_data['url']

# 2. Define our two types of features
# Put your best numerical features here
num_features = ['entropy', 'url_length', 'digits', 'path_length', 'suspicious_words', 'dot', 'has_dash'] 

# 3. Build the "Preprocessor" Engine
# This handles the text math and number math simultaneously 
preprocessor = ColumnTransformer(
    transformers=[
        # This breaks the raw URL string into 2, 3, and 4-character chunks!
        ('text', TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=2000), 'url'),
        
        # This lets your numerical features pass straight through
        ('num', 'passthrough', num_features)
    ]
)

# 4. Build the Ultimate Pipeline
# This connects the preprocessor directly to the Random Forest
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])


In [6]:

# 5. Split the Data
X = data[['url'] + num_features] # We need the raw url AND the numbers
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training the Ultimate NLP Pipeline... (This might take 1-3 minutes) 🚀")

Training the Ultimate NLP Pipeline... (This might take 1-3 minutes) 🚀


In [7]:

# 6. Train the entire pipeline!
pipeline.fit(X_train, y_train)

# 7. Evaluate
rf_nlp_pred = pipeline.predict(X_test)
rf_nlp_acc = accuracy_score(y_test, rf_nlp_pred)

print(f"\n🏆 Ultimate NLP Random Forest Accuracy: {rf_nlp_acc * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, rf_nlp_pred))

# Save this masterpiece
joblib.dump(pipeline, '../models/v2_models/nlp_random_forest.pkl')


🏆 Ultimate NLP Random Forest Accuracy: 97.32%

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.97      0.97      7840
           1       0.98      0.97      0.98     11343

    accuracy                           0.97     19183
   macro avg       0.97      0.97      0.97     19183
weighted avg       0.97      0.97      0.97     19183



['../models/v2_models/nlp_random_forest.pkl']

In [12]:
rf_nlp = confusion_matrix(y_test, rf_nlp_pred)
plt.figure(figsize=(6,4))
sns.heatmap(rf_nlp, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix for RF_NLP\nOverall Accuracy: {rf_nlp_acc * 100:.2f}%')
plt.savefig("../data/outputs/RF_NLP.png")
plt.close()